# Drone Flyby: synthetic data + YOLO training (Kaggle)

Settings (right-hand panel): **Accelerator = GPU T4 x2**, **Internet = on**.
Add the dataset `drone-flyby-code` (the zip with `training/`) via *Add Input*.
Run the cells top to bottom. At the end, download `best.pt` from the Output tab.

In [ ]:
# Same ultralytics version as the laptop, so best.pt loads in the service.
!pip install -q ultralytics==8.4.152

In [ ]:
# Official repo: the 25 Helsinki frames + utils.py/dtos.py the scripts import.
!git clone --depth 1 https://github.com/amboltio/Nordic-AI-Cup-2026.git /tmp/official
import glob, shutil
src = glob.glob('/kaggle/input/**/training/make_dataset.py', recursive=True)
assert src, 'Add the drone-flyby-code dataset as input'
shutil.copytree(src[0].rsplit('/', 1)[0], '/tmp/official/drone-flyby/training', dirs_exist_ok=True)
%cd /tmp/official/drone-flyby

In [ ]:
# 1. Cut the objects out.  2. Build the dataset (a few minutes).
!python training/extract_patches.py
!python training/make_dataset.py --scenes 800 --out /tmp/yolo

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # nano: ~110 ms per view on the laptop CPU
model.train(
    data='/tmp/yolo/data.yaml',
    imgsz=960,          # views are 960x540: no extra shrinking
    epochs=40,
    batch=16,
    device=0,
    cache='ram',        # PNG decoding is the bottleneck otherwise
    workers=4,
    # The drone flies at a fixed altitude, so objects barely change size.
    scale=0.1,
    degrees=0.0,        # rotation already done while pasting
    flipud=0.5,
    fliplr=0.5,
    project='/kaggle/working/runs',
    name='yolo11n_960',
    plots=True,
)

In [ ]:
import shutil
shutil.copy('/kaggle/working/runs/yolo11n_960/weights/best.pt', '/kaggle/working/best.pt')
print('Download best.pt from the Output tab')